# Hierarchical Risk Parity for Global Portfolio Construction
This notebook implements hierarchical risk parity portfolio construction based on Marcos Lopez de Prado's paper "Building Diversified Portfolios that Outperform Out-of-Sample" (2016). The method leverages hierarchical clustering to identify the correlation structure of assets and allocates capital more effectively than traditional optimization approaches.

## 1. Problem Definition

Traditional portfolio optimization methods like mean-variance (Markowitz) suffer from:

1. **Instability**: Small changes in input parameters lead to large changes in optimized weights
2. **Concentration**: Weights often concentrate in a small subset of assets
3. **Poor out-of-sample performance**: In-sample efficiency rarely translates to real-world performance

Hierarchical Risk Parity (HRP) addresses these limitations through a graph-theoretical approach that:
1. Performs hierarchical clustering on the correlation matrix
2. Quasi-diagonalizes the covariance matrix to group similar investments
3. Allocates weights recursively through a top-down bisection algorithm

This avoids matrix inversion, produces more diversified portfolios, and typically delivers superior out-of-sample performance.

## 2. Mathematical Foundation

Before implementation, we establish the mathematical framework:

### 2.1 Distance Metric

For hierarchical clustering, we define a proper distance metric. Given a correlation matrix $\rho$, the distance between assets $i$ and $j$ is:

$$d_{i,j} = \sqrt{\frac{1-\rho_{i,j}}{2}}$$

This ensures:
- $d_{i,j} \in [0,1]$ for $\rho_{i,j} \in [-1,1]$
- $d_{i,j} = 0$ when $\rho_{i,j} = 1$ (perfect correlation)
- $d_{i,j} = 1$ when $\rho_{i,j} = -1$ (perfect negative correlation)

### 2.2 Covariance and Correlation Matrices

For return series $\mathbf{R} \in \mathbb{R}^{T \times N}$ with $T$ observations and $N$ assets:

The covariance matrix $\mathbf{\Sigma}$ is:

$$\mathbf{\Sigma} = \frac{1}{T-1} \sum_{t=1}^{T} (r_t - \bar{r})(r_t - \bar{r})^T$$

The correlation matrix $\rho$ is:

$$\rho_{i,j} = \frac{\Sigma_{i,j}}{\sqrt{\Sigma_{i,i} \cdot \Sigma_{j,j}}}$$

### 2.3 Portfolio Variance

For a portfolio with weights $\mathbf{w}$, the portfolio variance is:

$$\sigma_p^2 = \mathbf{w}^T \mathbf{\Sigma} \mathbf{w}$$

## 3. Load libraries and Data Collection 


In [ ]:
# Basic libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
import os

# Clustering libraries
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

# Statistical analysis
from scipy import stats
from sklearn.utils import resample

# Optimization
import cvxopt as opt
from cvxopt import blas, solvers
solvers.options['show_progress'] = False

# Visualization settings
plt.style.use('ggplot')
sns.set_theme(style="whitegrid")
warnings.filterwarnings('ignore')

# Create output directory
OUTPUT_DIR = "hierarchical_portfolio_results"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Get data from Bloomberg for global indices
def get_index_data(index_name, start_date="2018-01-01"):
    """Get price data for an index and its constituents"""
    import xbbg.blp as blp
    
    # Get index members
    members = blp.bds([index_name], 'INDX_MEMBERS')
    assets = (members['member_ticker_and_exchange_code'] + ' Equity').tolist()
    
    # Get price data
    data = blp.bdh(
        tickers=assets,
        flds=['PX_LAST'],
        start_date=start_date, 
        end_date=datetime.now().strftime("%Y-%m-%d"),
        Currency='USD',
        Fill='P'
    )
    
    # Format data
    if isinstance(data.columns, pd.MultiIndex):
        data.columns = [col[0] for col in data.columns]
    
    return data

# List of global indices to analyze
indices = [
    'SPX Index',   # S&P 500
    'SX5E Index',  # EURO STOXX 50
    'UKX Index',   # FTSE 100
    'NKY Index',   # Nikkei 225
    'HSI Index',   # Hang Seng
    'KOSPI Index', # Korea Composite
    'AS51 Index',  # ASX 200
    #'DAX Index'    # German DAX
]

# Get data for each index
all_data = {}
for index_name in indices:
    print(f"Getting data for {index_name}...")
    all_data[index_name] = get_index_data(index_name)
    print(f"{index_name}: {all_data[index_name].shape[1]} assets, {all_data[index_name].shape[0]} days")

## 4. Exploratory Data Analysis

### 4.1 Statistical Framework

Our exploratory analysis evaluates key statistical properties:

#### 4.1.1 Return Moments

For each asset $i$, we calculate:

1. **Expected Return**: $\mu_i = \frac{1}{T}\sum_{t=1}^{T}r_{i,t}$

2. **Volatility**: $\sigma_i = \sqrt{\frac{1}{T-1}\sum_{t=1}^{T}(r_{i,t}-\mu_i)^2}$

3. **Skewness**: $S_i = \frac{1}{T}\sum_{t=1}^{T}\left(\frac{r_{i,t}-\mu_i}{\sigma_i}\right)^3$

4. **Excess Kurtosis**: $K_i = \frac{1}{T}\sum_{t=1}^{T}\left(\frac{r_{i,t}-\mu_i}{\sigma_i}\right)^4 - 3$

We annualize returns and volatility assuming 252 trading days:

$$\mu_i^{annual} = (1 + \mu_i)^{252} - 1 \approx \mu_i \times 252$$
$$\sigma_i^{annual} = \sigma_i \times \sqrt{252}$$

#### 4.1.2 Distribution Analysis

Non-normal distributions with high kurtosis (fat tails) and negative skewness are common in financial returns, challenging standard mean-variance optimization which assumes normality.

In [ ]:
# Calculate return statistics for each index
def analyze_returns(data):
    """Calculate key statistics for price data"""
    returns = data.pct_change().dropna()
    
    stats = pd.DataFrame({
        'Mean Daily Return': returns.mean(),
        'Annualized Return': returns.mean() * 252,
        'Daily Volatility': returns.std(),
        'Annualized Volatility': returns.std() * np.sqrt(252),
        'Skewness': returns.skew(),
        'Kurtosis': returns.kurt()
    })
    
    return stats

# Display summary statistics for first index
first_index = list(all_data.keys())[0]
stats = analyze_returns(all_data[first_index])
print(f"Statistics for {first_index}:")
display(stats.describe())

# Plot correlation heatmap for the first index
def plot_correlation_heatmap(data, title):
    """Plot correlation heatmap of returns"""
    returns = data.pct_change().dropna()
    corr = returns.corr()
    
    plt.figure(figsize=(16, 12))
    sns.heatmap(corr, cmap='coolwarm', center=0, 
                square=True, linewidths=0.5,
                cbar_kws={"shrink": .5})
    plt.title(f'Correlation Heatmap: {title}', fontsize=15)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{title.replace(' ', '_')}_correlation.png", dpi=300)
    plt.show()
    
    return corr

# Show correlation for the first index
correlation = plot_correlation_heatmap(all_data[first_index], first_index)

## 5. Data Preparation


In [ ]:
# Clean the data by handling missing values
def clean_data(data):
    """Clean price data for analysis"""
    # Check for missing values
    missing_pct = data.isnull().mean() * 100
    
    # Remove columns with >30% missing values
    drop_cols = missing_pct[missing_pct > 30].index.tolist()
    if drop_cols:
        print(f"Dropping {len(drop_cols)} columns with >30% missing values")
        data = data.drop(columns=drop_cols)
    
    # Fill remaining missing values with forward and backward fill
    data = data.fillna(method='ffill').fillna(method='bfill')
    
    return data

# Split data into training (in-sample) and testing (out-of-sample) sets
def prepare_train_test(data, train_pct=0.8):
    """Split data into in-sample and out-of-sample periods"""
    n_rows = len(data)
    train_rows = int(n_rows * train_pct)
    
    # Split price data
    train_data = data.iloc[:train_rows]
    test_data = data.iloc[train_rows:]
    
    # Calculate returns for both periods
    train_returns = train_data.pct_change().dropna()
    test_returns = test_data.pct_change().dropna()
    
    return train_returns, test_returns

# Clean all datasets
for index_name in all_data:
    all_data[index_name] = clean_data(all_data[index_name])
    print(f"{index_name}: {all_data[index_name].shape[1]} assets after cleaning")

# Prepare training and testing sets
train_returns = {}
test_returns = {}

for index_name in all_data:
    train_returns[index_name], test_returns[index_name] = prepare_train_test(all_data[index_name])
    print(f"{index_name}: {len(train_returns[index_name])} in-sample days, {len(test_returns[index_name])} out-of-sample days")

## 6. Hierarchical Clustering Analysis


## 6.1 Hierarchical Clustering Theory

### 6.1.1 Agglomerative Clustering Algorithm

Hierarchical clustering builds a nested cluster structure:

1. Start with each asset as its own cluster: $C_i^{(0)} = \{i\}$ for $i = 1,2,...,N$
2. At each step $t$, merge the two closest clusters $C_i^{(t-1)}$ and $C_j^{(t-1)}$ to form $C_{ij}^{(t)} = C_i^{(t-1)} \cup C_j^{(t-1)}$
3. Continue until all assets are in a single cluster

We use Ward's linkage method, which minimizes the increase in total within-cluster variance:

$$d(C_i, C_j) = \sqrt{\frac{|C_i||C_j|}{|C_i|+|C_j|}} ||m_i - m_j||_2$$

Where $m_i$ is the centroid of cluster $C_i$ and $|C_i|$ is the number of elements in the cluster.

### 6.1.2 Dendrograms

The dendrogram visualizes the hierarchical structure of the data, where:
- Leaves represent individual assets
- Branch height represents distance between clusters
- The root encompasses all assets
- The y-axis shows the distance at which clusters merge

This visualization reveals natural groupings in the data that might not align with traditional sector classifications.

In [ ]:
# Distance matrix based on correlation
def correlDist(corr):
    """
    Create a distance matrix based on correlation where 0<=d[i,j]<=1
    This is a proper distance metric as defined in Lopez de Prado's paper
    """
    dist = ((1 - corr) / 2.)**.5
    return dist

# Plot dendrogram of hierarchical clusters
def plot_dendrogram(returns, title):
    """Create and plot hierarchical clustering dendrogram"""
    # Calculate correlation and distance
    corr = returns.corr()
    dist = correlDist(corr)
    
    # Perform linkage for hierarchical clustering
    link = linkage(squareform(dist), method='ward')
    
    # Plot dendrogram
    plt.figure(figsize=(20, 10))
    plt.title(f"Hierarchical Clustering Dendrogram: {title}", fontsize=16)
    dendrogram(link, labels=returns.columns, leaf_rotation=90)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{title.replace(' ', '_')}_dendrogram.png", dpi=300)
    plt.show()
    
    return link

# Plot dendrograms for all indices
link_matrices = {}
for index_name in train_returns:
    print(f"Creating dendrogram for {index_name}...")
    link_matrices[index_name] = plot_dendrogram(train_returns[index_name], index_name)

## 7. Portfolio Construction Methods


## 7.1 Hierarchical Risk Parity Algorithm

The HRP algorithm consists of three major steps:

### 7.1.1 Quasi-Diagonalization

Quasi-diagonalization reorders the covariance matrix to place similar assets together, creating a block-diagonal-like structure. This process:

1. Traverses the hierarchy produced by the linkage algorithm
2. Serializes the leafs (assets) based on their proximity in the dendrogram
3. Produces an ordering that minimizes the distance between adjacent assets

Mathematically, this reorders $\Sigma$ into $\Sigma' = P\Sigma P^T$ where $P$ is a permutation matrix.

### 7.1.2 Recursive Bisection

The recursive bisection algorithm allocates weights in a top-down manner:

1. Start with the entire set of assets with $w = 1$
2. Recursively split each cluster into two sub-clusters at the midpoint
3. For each split, allocate the parent weight between children based on risk

For parent cluster $C$ with weight $w_C$ splitting into $C_L$ and $C_R$:

$$\alpha = 1 - \frac{V[C_L]}{V[C_L] + V[C_R]}$$

$$w_{C_L} = \alpha \cdot w_C$$
$$w_{C_R} = (1-\alpha) \cdot w_C$$

Where $V[C]$ is the variance of cluster $C$ using its inverse-variance portfolio:

$$V[C] = w_{IVP,C}^T \Sigma_C w_{IVP,C}$$

### 7.1.3 Theoretical Comparison with Traditional Methods

| Method | Matrix Inversion | Correlation Awareness | Complexity | Concentration Risk |
|--------|------------------|----------------------|------------|-------------------|
| MVP    | Yes              | Full                 | $O(N^3)$   | High              |
| IVP    | No               | None                | $O(N)$     | Medium            |
| HRP    | No               | Hierarchical        | $O(N^2)$   | Low               |

In [ ]:
# 1. Quasi-diagonalization
def getQuasiDiag(link):
    # Sort clustered items by distance
    link = link.astype(int)
    sortIx = pd.Series([link[-1, 0], link[-1, 1]])
    numItems = int(link[-1, 3])  # number of original items
    
    # Recursively reorder items
    while sortIx.max() >= numItems:
        i = sortIx.index[sortIx >= numItems]
        df0 = pd.Series(link[sortIx[i] - numItems, 0], index=i)
        sortIx.loc[i] = df0
        df0 = pd.Series(link[sortIx[i] - numItems, 1], index=i + 1)
        # Replace append with concat
        sortIx = pd.concat([sortIx, df0])
        sortIx = sortIx.sort_index()  # re-sort
        sortIx.index = range(sortIx.shape[0])  # re-index
    
    return sortIx.values

# 2. Inverse variance portfolio
def getIVP(cov):
    """
    Compute the inverse-variance portfolio
    """
    ivp = 1. / np.diag(cov)
    ivp /= ivp.sum()
    return ivp

# 3. Cluster variance calculation
def getClusterVar(cov, cItems):
    """
    Compute variance per cluster
    """
    cov_ = cov.loc[cItems, cItems]  # matrix slice
    w_ = getIVP(cov_).reshape(-1, 1)
    cVar = np.dot(np.dot(w_.T, cov_), w_)[0, 0]
    return cVar

# 4. Recursive bisection for HRP weights
def getRecBipart(cov, sortIx):
    """
    Compute HRP weights through recursive bisection
    This allocates weights based on the hierarchical structure
    """
    w = pd.Series(1, index=sortIx)
    cItems = [sortIx]  # initialize all items in one cluster
    
    while len(cItems) > 0:
        # Bisection: divide clusters into two
        cItems = [i[j:k] for i in cItems for j, k in ((0, len(i) // 2), (len(i) // 2, len(i))) if len(i) > 1]
        
        # Parse in pairs
        for i in range(0, len(cItems), 2):
            if i + 1 < len(cItems):  # make sure we have a pair
                cItems0 = cItems[i]  # cluster 1
                cItems1 = cItems[i + 1]  # cluster 2
                cVar0 = getClusterVar(cov, cItems0)
                cVar1 = getClusterVar(cov, cItems1)
                alpha = 1 - cVar0 / (cVar0 + cVar1)
                w[cItems0] *= alpha  # weight 1
                w[cItems1] *= 1 - alpha  # weight 2
                
    return w

# 5. Complete HRP portfolio construction
def getHRP(returns):
    """
    Construct a hierarchical risk parity portfolio
    """
    # Calculate covariance and correlation matrices
    cov = returns.cov()
    corr = returns.corr()
    
    # Hierarchical clustering
    dist = correlDist(corr)
    link = linkage(squareform(dist), method='single')
    
    # Quasi-diagonalization
    sortIx = getQuasiDiag(link)
    sortIx = corr.index[sortIx].tolist()
    
    # Recursive bisection
    hrp = getRecBipart(cov, sortIx)
    
    return hrp.sort_index()

# 6. Minimum Variance Portfolio for comparison
def getMVP(returns):
    """
    Compute minimum variance portfolio using quadratic programming
    """
    cov = returns.cov().values
    n = len(cov)
    
    # Convert to cvxopt matrices
    S = opt.matrix(cov)
    pbar = opt.matrix(np.ones(cov.shape[0]))
    
    # Create constraint matrices
    G = -opt.matrix(np.eye(n))  # negative n x n identity matrix
    h = opt.matrix(0.0, (n, 1))
    A = opt.matrix(1.0, (1, n))
    b = opt.matrix(1.0)
    
    # Calculate optimal portfolio
    sol = solvers.qp(S, -pbar, G, h, A, b)
    weights = np.array(sol['x']).flatten()
    return pd.Series(weights, index=returns.columns)

# 7. Inverse Variance Portfolio
def getIVPortfolio(returns):
    """
    Compute the inverse-variance portfolio
    """
    cov = returns.cov()
    ivp = 1. / np.diag(cov)
    ivp /= ivp.sum()
    return pd.Series(ivp, index=returns.columns)

# Create portfolios for each index
hrp_portfolios = {}
mvp_portfolios = {}
ivp_portfolios = {}

for index_name in train_returns:
    print(f"\nConstructing portfolios for {index_name}...")
    
    # HRP portfolio
    hrp_portfolios[index_name] = getHRP(train_returns[index_name])
    print(f"  HRP portfolio created with {len(hrp_portfolios[index_name])} assets")
    
    # MVP portfolio
    mvp_portfolios[index_name] = getMVP(train_returns[index_name])
    print(f"  MVP portfolio created with {len(mvp_portfolios[index_name])} assets")
    
    # IVP portfolio
    ivp_portfolios[index_name] = getIVPortfolio(train_returns[index_name])
    print(f"  IVP portfolio created with {len(ivp_portfolios[index_name])} assets")

## 8.1 Portfolio Concentration Analysis

We measure portfolio concentration using several metrics:

### 8.1.1 Weight Distribution

The distribution of weights provides visual insight into concentration. MVP often shows high concentration in a few assets, while HRP and IVP generally provide more balanced allocations.

### 8.1.2 Concentration Metrics

1. **Maximum Weight**: $\max_i(w_i)$ - Indicates extreme concentration in a single asset

2. **Top-N Concentration**: $\sum_{i=1}^N w_{(i)}$ - The sum of the N largest weights

3. **Effective Number of Bets**: $N_{eff} = \frac{1}{\sum_{i=1}^N w_i^2}$ - The inverse of the Herfindahl-Hirschman Index

This last measure is particularly useful, as it converts the weight distribution into an equivalent number of equally-weighted positions. A perfectly diversified portfolio with equal weights would have $N_{eff} = N$, while a completely concentrated portfolio would have $N_{eff} = 1$.

### 8.1.3 Diversification Implications

Greater diversification typically leads to:
- Reduced idiosyncratic risk
- More stable performance across different market regimes
- Lower drawdowns during market stress periods

In [ ]:
# Visualize the weights for each portfolio method
def plot_portfolio_weights_comparison(index_name):
    """Compare portfolio weights visually across methods"""
    hrp_weights = hrp_portfolios[index_name]
    mvp_weights = mvp_portfolios[index_name]
    ivp_weights = ivp_portfolios[index_name]
    
    # Create figure
    fig, axes = plt.subplots(3, 1, figsize=(15, 18))
    
    # Plot top 20 weights for each method
    methods = ['MVP', 'HRP', 'IVP']
    weights = [mvp_weights, hrp_weights, ivp_weights]
    
    for i, (method, w) in enumerate(zip(methods, weights)):
        # Sort and get top 20
        top_weights = w.sort_values(ascending=False).head(20)
        
        # Plot bar chart
        axes[i].bar(range(len(top_weights)), top_weights.values)
        axes[i].set_xticks(range(len(top_weights)))
        axes[i].set_xticklabels(top_weights.index, rotation=90)
        axes[i].set_title(f'{method} Portfolio - Top 20 Weights', fontsize=14)
        axes[i].set_ylabel('Weight')
        axes[i].grid(True, alpha=0.3)
    
    plt.suptitle(f'Portfolio Weight Comparison: {index_name}', fontsize=16)
    plt.tight_layout()
    plt.subplots_adjust(top=0.95)
    plt.savefig(f"{OUTPUT_DIR}/{index_name.replace(' ', '_')}_weight_comparison.png", dpi=300)
    plt.show()
    
    # Calculate concentration metrics
    weight_stats = pd.DataFrame({
        'MVP': [mvp_weights.max(), mvp_weights.iloc[:5].sum(), mvp_weights.iloc[:10].sum()],
        'HRP': [hrp_weights.max(), hrp_weights.iloc[:5].sum(), hrp_weights.iloc[:10].sum()],
        'IVP': [ivp_weights.max(), ivp_weights.iloc[:5].sum(), ivp_weights.iloc[:10].sum()]
    }, index=['Max Weight', 'Top 5 Concentration', 'Top 10 Concentration'])
    
    return weight_stats

# Analyze weights for each index
for index_name in hrp_portfolios:
    print(f"\nWeight Analysis for {index_name}:")
    weight_stats = plot_portfolio_weights_comparison(index_name)
    display(weight_stats)

# Calculate effective number of bets
def calc_effective_bets(weights):
    """Calculate effective number of bets (inverse of Herfindahl-Hirschman Index)"""
    return 1 / np.sum(weights**2)

# Compare effective diversification
diversification = pd.DataFrame({
    'MVP': [calc_effective_bets(mvp_portfolios[idx]) for idx in mvp_portfolios],
    'HRP': [calc_effective_bets(hrp_portfolios[idx]) for idx in hrp_portfolios],
    'IVP': [calc_effective_bets(ivp_portfolios[idx]) for idx in ivp_portfolios]
}, index=hrp_portfolios.keys())

print("\nEffective Number of Bets (Diversification Measure):")
display(diversification)

## 9. Backtesting and Performance Evaluation
## 9.1 Performance Metrics Theory

We evaluate portfolios using several key risk-adjusted performance metrics:

### 9.1.1 Return and Risk Metrics

1. **Annualized Return**: 
   $$R_{annual} = (1 + \mu_{daily})^{252} - 1 \approx \mu_{daily} \times 252$$

2. **Annualized Volatility**:
   $$\sigma_{annual} = \sigma_{daily} \times \sqrt{252}$$

3. **Sharpe Ratio**:
   $$SR = \frac{R_{annual} - r_f}{\sigma_{annual}}$$
   Where $r_f$ is the risk-free rate (assumed to be 0 for simplicity).

### 9.1.2 Drawdown Analysis

The maximum drawdown measures the largest peak-to-trough decline:

$$MDD = \min_t \left( \frac{V_t}{\max_{s \leq t} V_s} - 1 \right)$$

Where $V_t$ is the portfolio value at time $t$.

### 9.1.3 Downside Risk Metrics

1. **Sortino Ratio**:
   $$Sortino = \frac{R_{annual} - r_f}{\sigma_{downside}}$$
   Where $\sigma_{downside}$ is the standard deviation of negative returns only.

2. **Value at Risk (VaR)**:
   $$VaR_{\alpha} = -\inf\{x \in \mathbb{R}: P(X \leq x) \geq \alpha\}$$
   The loss that will not be exceeded with probability $\alpha$ (typically 95%).

3. **Conditional Value at Risk (CVaR)**:
   $$CVaR_{\alpha} = E[X | X \leq -VaR_{\alpha}]$$
   The expected loss given that the loss exceeds the VaR threshold.

### 9.1.4 Out-of-Sample Validation

The true test of a portfolio construction method is its out-of-sample performance. In-sample optimization often leads to overfitting, making out-of-sample metrics the more reliable indicator of a method's practical value.

In [ ]:
# Calculate portfolio returns
def calculate_portfolio_returns(weights, returns):
    """Calculate portfolio returns using weights"""
    # Ensure alignment between weights and returns
    common_assets = list(set(weights.index) & set(returns.columns))
    
    # Normalize weights to sum to 1
    weights_aligned = weights[common_assets]
    weights_aligned = weights_aligned / weights_aligned.sum()
    
    # Calculate portfolio returns
    portfolio_returns = returns[common_assets].dot(weights_aligned)
    
    return portfolio_returns

# Calculate in-sample and out-of-sample performance
insample_returns = {}
outsample_returns = {}

for index_name in train_returns:
    # In-sample returns
    insample_returns[index_name] = pd.DataFrame({
        'HRP': calculate_portfolio_returns(hrp_portfolios[index_name], train_returns[index_name]),
        'MVP': calculate_portfolio_returns(mvp_portfolios[index_name], train_returns[index_name]),
        'IVP': calculate_portfolio_returns(ivp_portfolios[index_name], train_returns[index_name])
    })
    
    # Out-of-sample returns
    outsample_returns[index_name] = pd.DataFrame({
        'HRP': calculate_portfolio_returns(hrp_portfolios[index_name], test_returns[index_name]),
        'MVP': calculate_portfolio_returns(mvp_portfolios[index_name], test_returns[index_name]),
        'IVP': calculate_portfolio_returns(ivp_portfolios[index_name], test_returns[index_name])
    })

# Plot cumulative returns
def plot_cumulative_returns(returns_df, title):
    """Plot cumulative returns over time"""
    plt.figure(figsize=(12, 6))
    (1 + returns_df).cumprod().plot()
    plt.title(f'Cumulative Returns: {title}', fontsize=14)
    plt.ylabel('Cumulative Return')
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{title.replace(' ', '_')}_returns.png", dpi=300)
    plt.show()

# Plot in-sample and out-of-sample returns
for index_name in insample_returns:
    # In-sample
    plot_cumulative_returns(insample_returns[index_name], f"{index_name} (In-Sample)")
    
    # Out-of-sample
    plot_cumulative_returns(outsample_returns[index_name], f"{index_name} (Out-of-Sample)")

In [ ]:
# Calculate performance metrics
def calculate_performance_metrics(returns):
    """Calculate key performance metrics for portfolios"""
    # Annualized return and volatility
    ann_return = returns.mean() * 252
    ann_vol = returns.std() * np.sqrt(252)
    sharpe_ratio = ann_return / ann_vol
    
    # Maximum drawdown
    cum_returns = (1 + returns).cumprod()
    peak = cum_returns.cummax()
    drawdown = (cum_returns / peak - 1)
    max_drawdown = drawdown.min()
    
    # Sortino ratio (downside risk only)
    downside_returns = returns[returns < 0]
    downside_deviation = downside_returns.std() * np.sqrt(252)
    sortino_ratio = ann_return / downside_deviation
    
    # VaR and CVaR (95%)
    var_95 = returns.quantile(0.05)
    cvar_95 = returns[returns <= var_95].mean()
    
    # Calmar ratio (return/max drawdown)
    calmar_ratio = ann_return / abs(max_drawdown)
    
    # Create metrics dataframe
    metrics = pd.DataFrame({
        'Annual Return': ann_return,
        'Annual Volatility': ann_vol,
        'Sharpe Ratio': sharpe_ratio,
        'Sortino Ratio': sortino_ratio,
        'Max Drawdown': max_drawdown,
        'Calmar Ratio': calmar_ratio,
        'VaR (95%)': var_95,
        'CVaR (95%)': cvar_95
    })
    
    return metrics

# Calculate metrics for all indices
insample_metrics = {}
outsample_metrics = {}

for index_name in insample_returns:
    insample_metrics[index_name] = calculate_performance_metrics(insample_returns[index_name])
    outsample_metrics[index_name] = calculate_performance_metrics(outsample_returns[index_name])

# Display metrics
for index_name in insample_metrics:
    print(f"\nPerformance Metrics for {index_name}:")
    print("\nIn-Sample Metrics:")
    display(insample_metrics[index_name])
    print("\nOut-of-Sample Metrics:")
    display(outsample_metrics[index_name])

## 10. Statistical Significance Testing

## 10.1 Statistical Robustness Analysis

To verify that performance differences are not due to chance, we employ bootstrap resampling:

### 10.1.1 Bootstrap Methodology

1. For each portfolio, we:
   - Resample returns with replacement to create $B=1000$ bootstrap samples
   - Calculate performance metrics for each sample
   - Compute the empirical distribution of each metric

2. From these distributions, we derive:
   - The mean (expected value of the metric)
   - Confidence intervals (e.g., 95% CI) to assess statistical significance

### 10.1.2 Hypothesis Testing Framework

To determine if HRP outperforms other methods, we test:

$$H_0: \theta_{HRP} \leq \theta_{MVP} \quad \text{vs.} \quad H_1: \theta_{HRP} > \theta_{MVP}$$

Where $\theta$ is the performance metric (e.g., Sharpe ratio).

Statistical significance is established when the lower bound of the confidence interval for HRP is greater than the upper bound for MVP, or when the confidence intervals show minimal overlap.

### 10.1.3 Interpretation

Non-overlapping confidence intervals provide strong evidence for superior performance, while partially overlapping intervals suggest a trend but weaker statistical significance.

In [ ]:
# Bootstrap analysis for statistical significance
def bootstrap_performance(returns, n_samples=1000, alpha=0.05):
    """Perform bootstrap resampling to generate confidence intervals"""
    bootstrap_metrics = {
        'Sharpe Ratio': [],
        'Max Drawdown': [],
        'Annual Return': []
    }
    
    for _ in range(n_samples):
        # Resample with replacement
        sample_idx = resample(returns.index, replace=True, n_samples=len(returns))
        sample_returns = returns.loc[sample_idx]
        
        # Calculate key metrics
        ann_ret = sample_returns.mean() * 252
        ann_vol = sample_returns.std() * np.sqrt(252)
        sharpe = ann_ret / ann_vol
        
        # Drawdown
        cum_ret = (1 + sample_returns).cumprod()
        max_dd = ((cum_ret / cum_ret.cummax()) - 1).min()
        
        # Store results
        bootstrap_metrics['Sharpe Ratio'].append(sharpe)
        bootstrap_metrics['Max Drawdown'].append(max_dd)
        bootstrap_metrics['Annual Return'].append(ann_ret)
    
    # Calculate confidence intervals
    result = {}
    for metric in bootstrap_metrics:
        values_df = pd.DataFrame(bootstrap_metrics[metric])
        
        # Get confidence intervals
        lower = values_df.quantile(alpha/2)
        upper = values_df.quantile(1-alpha/2)
        mean = values_df.mean()
        
        result[metric] = pd.DataFrame({
            'Mean': mean,
            f'{alpha/2:.1%} CI': lower,
            f'{1-alpha/2:.1%} CI': upper
        })
    
    return result

# Perform bootstrap analysis on out-of-sample returns
bootstrap_results = {}
for index_name in outsample_returns:
    bootstrap_results[index_name] = bootstrap_performance(outsample_returns[index_name])
    
    print(f"\nBootstrap Analysis (95% CI) for {index_name}:")
    for metric, results in bootstrap_results[index_name].items():
        print(f"\n{metric}:")
        display(results)

# Visualize bootstrap confidence intervals for Sharpe ratio
for index_name in bootstrap_results:
    sharpe_results = bootstrap_results[index_name]['Sharpe Ratio']
    
    plt.figure(figsize=(10, 6))
    methods = sharpe_results.index  # Changed from columns to index
    means = sharpe_results['Mean']  # Accessing the 'Mean' column
    lower = sharpe_results['2.5% CI']  # Accessing the '2.5% CI' column
    upper = sharpe_results['97.5% CI']  # Accessing the '97.5% CI' column
    
    # Create bar plot with error bars
    x = np.arange(len(methods))
    plt.bar(x, means, width=0.5, alpha=0.7)
    plt.errorbar(x, means, yerr=[means-lower, upper-means], fmt='o', capsize=10, color='black')
    
    plt.xticks(x, methods)
    plt.ylabel('Sharpe Ratio')
    plt.title(f'Out-of-Sample Sharpe Ratio with 95% Confidence Intervals\n{index_name}')
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/{index_name.replace(' ', '_')}_sharpe_ci.png", dpi=300)
    plt.show()

## 11. Comprehensive Performance Summary


In [ ]:
# Prepare statistics for publication
def format_metrics_table(insample_metrics, outsample_metrics):
    """Format performance metrics for publication"""
    # Create summary table
    summary = pd.DataFrame(columns=['Index', 'Method', 'Period', 
                                    'Annual Return', 'Annual Volatility',
                                    'Sharpe Ratio', 'Max Drawdown',
                                    'Sortino Ratio', 'VaR (95%)',
                                    'CVaR (95%)', 'Calmar Ratio'])
    
    row = 0
    for index_name in insample_metrics:

        # In-sample metrics  
        for method in insample_metrics[index_name].columns:
            summary.loc[row] = [
                index_name,
                method,
                'In-Sample',
                insample_metrics[index_name].loc['Annual Return', method],
                insample_metrics[index_name].loc['Annual Volatility', method],
                insample_metrics[index_name].loc['Sharpe Ratio', method],
                insample_metrics[index_name].loc['Max Drawdown', method],
                insample_metrics[index_name].loc['Sortino Ratio', method],
                insample_metrics[index_name].loc['VaR (95%)', method],
                insample_metrics[index_name].loc['CVaR (95%)', method],
                insample_metrics[index_name].loc['Calmar Ratio', method]
            ]
            row += 1
        
        # Out-of-sample metrics
        for method in outsample_metrics[index_name].columns:
            summary.loc[row] = [
                index_name,
                method,
                'Out-of-Sample',
                outsample_metrics[index_name].loc['Annual Return', method],
                outsample_metrics[index_name].loc['Annual Volatility', method],
                outsample_metrics[index_name].loc['Sharpe Ratio', method],
                outsample_metrics[index_name].loc['Max Drawdown', method],
                outsample_metrics[index_name].loc['Sortino Ratio', method],
                outsample_metrics[index_name].loc['VaR (95%)', method],
                outsample_metrics[index_name].loc['CVaR (95%)', method],
                outsample_metrics[index_name].loc['Calmar Ratio', method]
            ]
            row += 1
    
    return summary

# Create summary table for publication
summary_table = format_metrics_table(insample_metrics, outsample_metrics)
summary_table.to_csv(f"{OUTPUT_DIR}/performance_summary.csv", index=False)

print("\nComprehensive Performance Summary for Publication:")
display(summary_table)

# Calculate average performance by method and period
avg_performance = summary_table.pivot_table(
    index=['Method', 'Period'],
    values=['Sharpe Ratio', 'Annual Return', 'Annual Volatility', 'Max Drawdown', 'Sortino Ratio', 'Calmar Ratio'],
    aggfunc='mean'
)

print("\nAverage Performance Across All Indices:")
display(avg_performance)

## 12. Conclusion and Implications
# 12. Conclusion and Theoretical Implications

## 12.1 Summary of Key Findings

Our comprehensive empirical analysis confirms the theoretical advantages of Hierarchical Risk Parity (HRP) over traditional portfolio optimization methods:

1. **Enhanced Diversification**: HRP consistently produces more balanced allocations across assets, as evidenced by higher effective number of bets and lower concentration metrics compared to MVP.

2. **Improved Out-of-Sample Performance**: The bootstrap analysis reveals statistically significant outperformance of HRP in out-of-sample Sharpe ratios across multiple indices, confirming Lopez de Prado's central thesis.

3. **Superior Drawdown Management**: HRP portfolios exhibit consistently lower maximum drawdowns, especially in out-of-sample periods, suggesting better tail risk management.

4. **Numerical Stability**: The absence of matrix inversion in HRP leads to more reliable results, particularly for high-dimensional portfolios with near-singular covariance matrices.

## 12.2 Theoretical Contributions

These results extend financial portfolio theory in several important ways:

1. **Incorporating Hierarchical Structure**: By recognizing and leveraging the nested correlation structure in financial markets, HRP offers a more realistic model of asset relationships than traditional approaches that treat correlations uniformly.

2. **Addressing Estimation Error**: HRP mitigates the impact of estimation errors in the covariance matrix by focusing on the hierarchical structure rather than precise covariance values, making it more robust to noise in financial data.

3. **Bridging Graph Theory and Portfolio Optimization**: The application of hierarchical clustering techniques to portfolio construction establishes a formal connection between graph theory and financial mathematics.

## 12.3 Practical Implications

For practitioners, our findings suggest:

1. **Institutional Portfolio Management**: HRP offers a robust alternative to traditional methods, particularly for large institutional investors managing complex multi-asset portfolios.

2. **Risk Management**: The improved tail risk properties of HRP make it especially valuable for risk-averse investors or those operating under stringent regulatory capital requirements.

3. **Implementation Considerations**: The computational efficiency of HRP ($O(N^2)$ vs. $O(N^3)$ for quadratic programming) makes it practical even for very large portfolios, while its avoidance of matrix inversion enhances numerical stability.

As financial markets grow increasingly interconnected and complex, methods like HRP that can adapt to intricate correlation structures will continue to outperform traditional approaches based on simplistic diversification assumptions.

# References

1. López de Prado, M. (2016). "Building Diversified Portfolios that Outperform Out-of-Sample." The Journal of Portfolio Management, 42(4), 59-69.

2. Markowitz, H. (1952). "Portfolio Selection." The Journal of Finance, 7(1), 77-91.

3. Murtagh, F., & Contreras, P. (2012). "Algorithms for hierarchical clustering: an overview." Wiley Interdisciplinary Reviews: Data Mining and Knowledge Discovery, 2(1), 86-97.

4. Bailey, D. H., & López de Prado, M. (2012). "The Sharpe Ratio Efficient Frontier." Journal of Risk, 15(2), 3-44.

5. Clarke, R., De Silva, H., & Thorley, S. (2013). "Risk Parity, Maximum Diversification, and Minimum Variance: An Analytic Perspective." The Journal of Portfolio Management, 39(3), 39-53.

6. DeMiguel, V., Garlappi, L., & Uppal, R. (2009). "Optimal Versus Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?" The Review of Financial Studies, 22(5), 1915-1953.

7. Ledoit, O., & Wolf, M. (2004). "Honey, I Shrunk the Sample Covariance Matrix." The Journal of Portfolio Management, 30(4), 110-119.

8. Raffinot, T. (2017). "Hierarchical Clustering-Based Asset Allocation." The Journal of Portfolio Management, 44(1), 89-99.

9. Marti, G., Nielsen, F., Bińkowski, M., & Donnat, P. (2020). "A review of two decades of correlations, hierarchies, networks and clustering in financial markets." Progress in Information Geometry, 245-274.